####HR:Find consecutive days of employee attendance

An HR team wants to find employees who have attended work for at least 3 consecute days.

Goal: Find employees who have at least 3 consecutive days of attendance.


In [0]:
# Create the attendance table in Unity Catalog
spark.sql("""
CREATE TABLE IF NOT EXISTS interview.`25_sql`.attendance (
  emp_id   INT,
  att_date DATE,
  status   STRING
)
""")

# Insert sample attendance records
spark.sql("""
INSERT INTO interview.`25_sql`.attendance VALUES
  (101, DATE '2024-01-01', 'Present'),
  (101, DATE '2024-01-02', 'Present'),
  (101, DATE '2024-01-03', 'Present'),
  (101, DATE '2024-01-04', 'Absent'),
  (101, DATE '2024-01-05', 'Present'),
  (101, DATE '2024-01-06', 'Present'),
  (101, DATE '2024-01-07', 'Present'),
  (102, DATE '2024-01-01', 'Present'),
  (102, DATE '2024-01-02', 'Absent'),
  (102, DATE '2024-01-03', 'Present'),
  (102, DATE '2024-01-04', 'Present'),
  (103, DATE '2024-01-01', 'Present'),
  (103, DATE '2024-01-02', 'Present'),
  (103, DATE '2024-01-03', 'Absent'),
  (103, DATE '2024-01-04', 'Present'),
  (103, DATE '2024-01-05', 'Present'),
  (103, DATE '2024-01-06', 'Present'),
  (104, DATE '2024-01-01', 'Present'),
  (104, DATE '2024-01-02', 'Present')
""")

# Verify the inserted records
display(spark.sql("SELECT * FROM interview.`25_sql`.attendance ORDER BY emp_id, att_date"))


Explanation: We use the ROW_NUMBER() pick to group consecutive date.
if the difference between date and its row number in the same those dates are consecutative 

Tip : This pattern is useful for finding consecutive date,login streaks,
attendance or activity series in real world scenario.

In [0]:
%sql
WITH att AS (
	SELECT 
		emp_id,
		att_date,	
		DATE_SUB(att_date,ROW_NUMBER() OVER(PARTITION BY emp_id ORDER BY att_date)) AS grp
FROM interview.`25_sql`.attendance
WHERE status='Present'
)
SELECT emp_id,
       MIN(att_date) AS start_date,
       MAX(att_date) AS end_date,
       COUNT(*) AS consecutive_days
FROM att
GROUP BY emp_id,grp
HAVING COUNT(*)>=3
ORDER BY emp_id,start_date;